In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
from snowflake.ml.model import task
import joblib, pathlib, tempfile, sklearn

session = get_active_session()
print("notebook sklearn:", sklearn.__version__)   # check before trusting the load

STAGE_FILE = ("@DEMO.GUARDANT_DEMO.GUARDANT_ENABLEMENT_REPO"
              "/branches/main/model/variant_clf.joblib")

with tempfile.TemporaryDirectory() as tmp:
    session.file.get(STAGE_FILE, tmp)          # Snowflake fetches it, not you
    clf = joblib.load(next(pathlib.Path(tmp).glob("*.joblib")))
print("loaded", type(clf).__name__)

# Signature from the real table rather than an undefined placeholder
sample = session.sql("""
    SELECT VAF, READ_DEPTH, ALT_READ_COUNT, MAPPING_QUALITY
    FROM DEMO.GUARDANT_DEMO.VARIANT_CALLS
    WHERE CALL_FILTER = 'PASS' LIMIT 100
""").to_pandas()

session.sql("USE DATABASE DEMO").collect()
session.sql("USE SCHEMA GUARDANT_DEMO").collect()

reg = Registry(session=session, database_name="DEMO", schema_name="GUARDANT_DEMO")

mv = reg.log_model(
    clf,
    model_name="GUARDANT_VARIANT_CLF",
    sample_input_data=sample,
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
    metrics={"roc_auc": 0.69},
    comment="RandomForest, pulled from the git stage inside Snowflake",
    target_platforms=["WAREHOUSE"],             # container runtime defaults to SPCS only
)
funcs = mv.show_functions()
print("registered V4:", [f["name"] if isinstance(f, dict) else f.name for f in funcs])